# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library and references all key fields using their `@id` as per the Croissant schema.

### Dataset Source
The dataset source is provided via its Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
ds = mlc.Dataset(croissant_url)

# Print dataset name and description
meta = ds.metadata
print(f"Dataset title: {meta.name}")
print(f"Description: {meta.description}\n")

## 2. Data Overview
Review available record sets, their fields, field types, and their `@id`. All references use the Croissant `@id`.

- List all record sets and their IDs
- For each record set, list fields/columns and their IDs


In [ ]:
# List all record sets by @id
print("Record sets available in this dataset:")
record_set_ids = [rs['@id'] for rs in ds.metadata.to_json().get('recordSet', [])]
for rs in ds.metadata.to_json().get('recordSet', []):
    rsid = rs['@id']
    rsname = rs.get('name', '(no name)')
    print(f"  - recordSet @id: '{rsid}' | name: {rsname}")

if not record_set_ids:
    print("No explicit recordSet definitions found in the metadata. Attempting to infer from schema...")
    # For some datasets, to_json may not expose a populated recordSet[] but the dataset API works.

# For demonstration, inspect the list of available record sets via the API
print("\nIterating over record sets using the Dataset.records(...):")
# This is usually not necessary but useful for datasets with a singular recordSet
try:
    # Try to find one record set and print a few records
    it = ds.records()
    for i, rec in enumerate(it):
        print(f"Row {i+1}: {{k: v for k,v in rec.items() if not k.startswith('_')}}")
        if i > 3:
            break
except Exception as e:
    print("Error extracting records: ", e)

# If explicit record set IDs are present, print the fields
for rs in ds.metadata.to_json().get('recordSet', []):
    rsid = rs['@id']
    print(f"\nFields for recordSet '{rsid}':")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for f in fields:
        fid = f['@id'] if isinstance(f, dict) else f
        print(f"  - field @id: {fid}")

## 3. Data Extraction
Load all data from the primary record set into a DataFrame for analysis. All `@id` references are used from the previous overview step.


In [ ]:
# Identify main record set (if only one is present, it's used)
record_set_ids = [rs["@id"] for rs in ds.metadata.to_json().get('recordSet', [])]
if not record_set_ids:
    # Fallback: Some datasets define only one record set (default intermediary)
    print("No explicit recordSet found, using default (None)")
    main_record_set_id = None
else:
    main_record_set_id = record_set_ids[0]

# Load records from the main record set
records = list(ds.records(record_set=main_record_set_id))
df = pd.DataFrame(records)

print("DataFrame columns (keys):")
print(df.columns.tolist())

# Show the first rows
df.head()

## 4. Exploratory Data Analysis (EDA)
We demonstrate filtering, normalization, and grouping using the field Croissant `@id`s as column keys. You may change these parameters based on fields available in the dataset.


In [ ]:
# List numeric candidate fields (by column name)
print("Numeric columns detected:")
numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
print(numeric_fields)

# For demonstration, try first numeric field
if numeric_fields:
    numeric_field_id = numeric_fields[0]  # This is the '@id' of the field in the schema/columns
else:
    raise Exception("No numeric fields found for EDA.")

print(f"Using numeric field (Croissant @id): {numeric_field_id}")

threshold = df[numeric_field_id].mean()

# Filtering
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records (z-score):")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field if available
categorical_fields = df.select_dtypes(include=['object']).columns.tolist()
if categorical_fields:
    group_field_id = categorical_fields[0]
    print(f"\nGrouping by: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(grouped_df.head())
else:
    print('No categorical fields to group by.')

## 5. Visualization
We demonstrate basic data visualizations using column `@id` as references, such as the distribution of the primary numeric field and grouped means.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Numeric field histogram
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# Boxplot by group if available
if categorical_fields:
    plt.figure(figsize=(10,4))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"Boxplot of {numeric_field_id} grouped by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45, ha='right')
    plt.show()

## 6. Conclusion
We demonstrated exploratory data analysis using Croissant dataset and `mlcroissant`. Fields and record sets are referenced by their Croissant `@id`. Data was loaded, filtered, normalized, grouped by a categorical key, and visualized.

- Review the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) and its Croissant schema for all possible `@id`s and data structure details.
- For more advanced workflows (merging with other datasets, inspecting provenance, etc.), continue to use the schema to refer to elements by `@id`.

Happy data exploration!